# 02 模块分开配置

目标：把架构配置、分词器、模型加载参数和生成参数拆开，理解部署时每类配置控制什么。


## 运行环境准备

这个 notebook 默认使用 `Qwen/Qwen2.5-0.5B-Instruct`，适合在魔搭 Notebook 里快速学习。

如果你想用更大的模型，可以把 `MODEL_ID` 改成 `Qwen/Qwen2.5-7B-Instruct`，然后重启内核重新运行。


In [1]:
from pathlib import Path

requirements_path = Path("requirements.txt")
if not requirements_path.exists():
    requirements_path = Path("../requirements.txt")

%pip install -r {requirements_path}



[notice] A new release of pip is available: 26.1 -> 26.1.1
[notice] To update, run: python3 -m pip install --upgrade pip
ERROR: Could not open requirements file: [Errno 2] No such file or directory: '../requirements.txt'
Note: you may need to restart the kernel to use updated packages.


In [2]:
import os
from pathlib import Path

MODEL_ID = os.getenv("MODEL_ID", "Qwen/Qwen2.5-0.5B-Instruct")
MODEL_SOURCE = os.getenv("MODEL_SOURCE", "modelscope").lower()


def resolve_model_path(model_id):
    if Path(model_id).exists():
        return model_id
    if MODEL_SOURCE != "modelscope":
        return model_id

    from modelscope import snapshot_download
    return snapshot_download(model_id)


MODEL_PATH = resolve_model_path(MODEL_ID)
print("MODEL_ID =", MODEL_ID)
print("MODEL_PATH =", MODEL_PATH)


/usr/local/lib/python3.12/dist-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


2026-05-19 22:40:30,218 - modelscope - INFO - Target directory already exists, skipping creation.


MODEL_ID = Qwen/Qwen2.5-0.5B-Instruct
MODEL_PATH = /mnt/workspace/.cache/modelscope/models/Qwen/Qwen2___5-0___5B-Instruct


## 1. 架构配置

`AutoConfig` 保存层数、hidden size、attention heads、模型类型等结构信息。


In [3]:
from transformers import (
    AutoConfig,
    AutoModelForCausalLM,
    AutoTokenizer,
    GenerationConfig,
)


config = AutoConfig.from_pretrained(MODEL_PATH, trust_remote_code=True)
print("model_type:", config.model_type)
print("hidden_size:", getattr(config, "hidden_size", "unknown"))
print("num_hidden_layers:", getattr(config, "num_hidden_layers", "unknown"))
print("num_attention_heads:", getattr(config, "num_attention_heads", "unknown"))
print("num_key_value_heads:", getattr(config, "num_key_value_heads", "unknown"))


model_type: qwen2
hidden_size: 896
num_hidden_layers: 24
num_attention_heads: 14
num_key_value_heads: 2


## 2. 分词器和模型加载

部署时常见的关键项是 `device_map`、`torch_dtype`、`pad_token` 和 `eos_token`。


In [4]:
tokenizer = AutoTokenizer.from_pretrained(
    MODEL_PATH,
    padding_side="left",
    trust_remote_code=True,
)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    MODEL_PATH,
    config=config,
    torch_dtype="auto",
    device_map="auto",
    trust_remote_code=True,
)


Loading weights: 100%|██████████| 290/290 [00:00<00:00, 534.00it/s]


## 3. 生成配置

`GenerationConfig` 管 max_new_tokens、temperature、top_p、重复惩罚和停止 token。


In [5]:
generation_config = GenerationConfig.from_pretrained(MODEL_PATH)
generation_config.max_new_tokens = 120
generation_config.do_sample = True
generation_config.temperature = 0.7
generation_config.top_p = 0.9
generation_config.repetition_penalty = 1.05
generation_config.pad_token_id = tokenizer.pad_token_id
generation_config.eos_token_id = tokenizer.eos_token_id

generation_config


GenerationConfig {
  "bos_token_id": 151643,
  "do_sample": true,
  "eos_token_id": 151645,
  "max_new_tokens": 120,
  "pad_token_id": 151643,
  "repetition_penalty": 1.05,
  "temperature": 0.7,
  "top_k": 20,
  "top_p": 0.9
}

## 4. 运行生成

这里开始能清楚看到：prompt 渲染、tokenize、generate、decode 是四个独立步骤。


In [6]:
messages = [
    {"role": "system", "content": "你是一个面试辅导老师。"},
    {"role": "user", "content": "解释 prefill 和 decode 的区别。"},
]

prompt = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True,
)

inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
outputs = model.generate(
    **inputs,
    generation_config=generation_config,
)

new_token_ids = outputs[0][inputs["input_ids"].shape[-1] :]
answer = tokenizer.decode(new_token_ids, skip_special_tokens=True)

print(answer)


在编程和数据处理领域中，预填充（prefill）和解码（decode）都是与编码（coding）相关的概念。它们分别用于不同的场景。

1. **预填充**（Prefill）：预填充是一种编程技术，用于在程序执行前，将一些常量或初始化值预先插入到程序的代码中。这通常是在编写代码之前进行的，目的是减少执行代码时的计算量。例如，在计算某些复杂公式时，如果事先知道结果，可以在执行之前计算出结果并存储下来，这样在后续使用时可以


你这段代码最适合做三类测试：dtype、device_map、tokenizer padding。建议你在 notebook 里把加载逻辑封装成函数，这样每次改参数不用重写。

In [7]:
import torch
from transformers import AutoConfig, AutoModelForCausalLM, AutoTokenizer

def load_model_for_test(
    model_path,
    torch_dtype="auto",
    device_map="auto",
    padding_side="left",
):
    config = AutoConfig.from_pretrained(
        model_path,
        trust_remote_code=True,
    )

    tokenizer = AutoTokenizer.from_pretrained(
        model_path,
        padding_side=padding_side,
        trust_remote_code=True,
    )

    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token

    model = AutoModelForCausalLM.from_pretrained(
        model_path,
        config=config,
        torch_dtype=torch_dtype,
        device_map=device_map,
        trust_remote_code=True,
    )

    return tokenizer, model, config

然后做测试：

In [8]:
tokenizer, model, config = load_model_for_test(
    MODEL_PATH,
    torch_dtype="auto",
    device_map="auto",
    padding_side="left",
)

Loading weights: 100%|██████████| 290/290 [00:00<00:00, 508.10it/s]


看模型实际 dtype 和设备：

In [9]:
param = next(model.parameters())
print("dtype:", param.dtype)
print("device:", param.device)
print("device_map:", getattr(model, "hf_device_map", None))

dtype: torch.bfloat16
device: cuda:0
device_map: None


你可以测试这些 dtype：

In [10]:
tests = [
    ("auto", "auto"),
    (torch.float16, "auto"),
    (torch.bfloat16, "auto"),
    (torch.float32, "auto"),
]

for dtype, device_map in tests:
    print("=" * 60)
    print("torch_dtype:", dtype)

    tokenizer, model, config = load_model_for_test(
        MODEL_PATH,
        torch_dtype=dtype,
        device_map=device_map,
    )

    param = next(model.parameters())
    print("actual dtype:", param.dtype)
    print("device:", param.device)
    print("device_map:", getattr(model, "hf_device_map", None))

    del model
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

torch_dtype: auto


Loading weights: 100%|██████████| 290/290 [00:00<00:00, 588.67it/s]


actual dtype: torch.bfloat16
device: cuda:0
device_map: None
torch_dtype: torch.float16


Loading weights: 100%|██████████| 290/290 [00:00<00:00, 2215.42it/s]


actual dtype: torch.float16
device: cuda:0
device_map: None
torch_dtype: torch.bfloat16


Loading weights: 100%|██████████| 290/290 [00:00<00:00, 569.25it/s]


actual dtype: torch.bfloat16
device: cuda:0
device_map: None
torch_dtype: torch.float32


Loading weights: 100%|██████████| 290/290 [00:00<00:00, 1574.21it/s]


actual dtype: torch.float32
device: cuda:0
device_map: None


常见区别：

torch.float32
精度高，但显存占用最大，推理慢。

torch.float16
显存约为 float32 的一半，GPU 推理常用。

torch.bfloat16
显存也约为 float32 的一半，数值范围比 float16 稳定；A100/H100/部分新卡更适合。

"auto"
让 transformers 根据模型权重和设备自动选择，最省心。

还可以测 padding_side：

In [11]:
for side in ["left", "right"]:
    tokenizer, model, config = load_model_for_test(
        MODEL_PATH,
        padding_side=side,
    )

    batch = tokenizer(
        ["你好", "请用三句话解释 KV cache 是什么"],
        padding=True,
        return_tensors="pt",
    )

    print("=" * 40)
    print("padding_side:", side)
    print(batch["input_ids"])
    print(batch["attention_mask"])

Loading weights: 100%|██████████| 290/290 [00:00<00:00, 577.88it/s]


padding_side: left
tensor([[151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 108386],
        [ 14880,  11622,  44991, 100908, 104136,  84648,   6500,  54851,  99245]])
tensor([[0, 0, 0, 0, 0, 0, 0, 0, 1],
        [1, 1, 1, 1, 1, 1, 1, 1, 1]])


Loading weights: 100%|██████████| 290/290 [00:00<00:00, 596.69it/s]


padding_side: right
tensor([[108386, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643],
        [ 14880,  11622,  44991, 100908, 104136,  84648,   6500,  54851,  99245]])
tensor([[1, 0, 0, 0, 0, 0, 0, 0, 0],
        [1, 1, 1, 1, 1, 1, 1, 1, 1]])


因为批量生成时，右侧是最新 token 位置，对 decoder-only 模型更自然。

你也可以测试生成速度：

In [12]:
from time import perf_counter

def test_generation(tokenizer, model):
    messages = [
        {"role": "user", "content": "用三句话解释 tokenizer 的作用。"}
    ]

    prompt = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
    )

    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

    start = perf_counter()
    outputs = model.generate(
        **inputs,
        max_new_tokens=100,
        do_sample=False,
    )
    elapsed = perf_counter() - start

    new_tokens = outputs[0][inputs["input_ids"].shape[-1]:]
    print("new tokens:", len(new_tokens))
    print("seconds:", round(elapsed, 3))
    print("tokens/s:", round(len(new_tokens) / elapsed, 2))
    print(tokenizer.decode(new_tokens, skip_special_tokens=True))

In [22]:
tokenizer, model, config = load_model_for_test(
    MODEL_PATH,
    torch_dtype="float8",
    device_map="auto",
    padding_side="left",
)

ValueError: `dtype` provided as a `str` can only be `'auto'`, or a string representation of a valid `torch.dtype`

In [20]:
test_generation(tokenizer, model)

new tokens: 95
seconds: 1.631
tokens/s: 58.26
1. tokenizer 是自然语言处理中的一个重要组件，用于将文本分割成单词、短语或句子的基本单位。
2. 它在机器翻译和语音识别等任务中起着关键作用，能够有效地将复杂的文本信息分解为更易于理解和处理的单元。
3. tokenizer 通过其强大的功能和灵活性，使得机器学习模型能够在处理大量文本数据时更加高效和准确地进行分类、聚类和预测等操作。


重点观察：

1. 模型是否能成功加载
2. actual dtype 是什么
3. GPU 显存占用
4. 首 token 延迟
5. tokens/s
6. 输出质量是否异常